<table>
<tr>                                                                                   
     <th>
         <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'><a href="https://www.kaggle.com/datasets/alicenkbaytop/abalone-dataset/data">Jeu de données Ormeau(Abalone)<br>Analyse exploratoire des données</a></div>
     </th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/abalone.jpg" width="96"></th>
 </tr>
<tr>                                                                                   
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/abalone01.jpg" width="512"></th>
 </tr>    
</table>


<div style='padding:15px;color:#030aa7;font-size:100%;text-align: left;font-family: Georgia, serif'><a href="https://archive.ics.uci.edu/dataset/1/abalone">Veuillez vous référer à la page UC Irvine Machine Learning Repository officielle pour plus de détails.</a></div>

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Introduction</div></b>
## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Import libriries </div></b>

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, warnings, os
from datetime import datetime as dt
from matplotlib import pyplot as plt

import matplotlib.font_manager as fm
import plotly.express as px
import plotly.graph_objs as go
# import graphviz, pydotplus
from IPython.display import Image
import re

font1 = fm.FontProperties(size=20)
font2 = fm.FontProperties(size=24)

warnings.filterwarnings(action="ignore")

if int(str(sns.__version__).split('.')[1]) > 8 : 
    plt.style.use('seaborn-v0_8-darkgrid')
else:
    plt.style.use('seaborn-darkgrid')
sns.set(font_scale=3)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.tree import DecisionTreeClassifier,export_graphviz

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Outils du document</div></b>

In [ ]:
palette = [ "#030aa7", "#e50000", "#d8863b", "#005f6a", "#6b7c85", "#751973", 
            "#0485d1", "#ff7855", "#fbeeac", "#0cb577", "#95a3a6", "#c071fe", 
            "#d1e5f0", "#fddbc7", "#ffffcb", "#12e193", "#d8dcd6", "#dfc5fe", 
          ]
sns.palplot(sns.color_palette(palette))

In [ ]:
repertoireRacine  = '.'
nomProjet         = "Abalone-Analyse exploratoire des données"

repertoireProjet  = os.path.join(repertoireRacine, nomProjet)
repertoireDonnees = os.path.join(repertoireProjet, 'repertoire.donnees')
repertoireImages  = os.path.join(repertoireProjet, 'repertoire.images')


def controleExistenceRepertoire( repertoire, create_if_needed=True):
    """Voir si le répertoire existe. S'il n'existe pas il est créé."""
    path_exists = os.path.exists(repertoire)
    if path_exists:
        if not os.path.isdir(repertoire):
            raise Exception("Trouvé le nom  "+repertoire +" mais c'est un fichier, pas un répertoire")
            # return False
        return True
    if create_if_needed:
        os.makedirs(repertoire)

def sauvegarderImage( fichier):
    """Enregistrez la figure. Appelez la méthode juste avant plt.show ()."""
    controleExistenceRepertoire(repertoireImages)
    plt.savefig(os.path.join(repertoireImages,
                             fichier+f"--{dt.now().strftime('%Y_%m_%d_%H.%M.%S')}.png"), 
                             dpi=600, 
                             bbox_inches='tight')

def sauvegarderImageSNS( sns_plot, fichier):
    """Enregistrez la figure. Appelez la méthode juste avant plt.show ()."""
    controleExistenceRepertoire(repertoireImages)
    fig = sns_plot.get_figure()
    fig.savefig(os.path.join(repertoireImages,fichier+'.png'))
    
controleExistenceRepertoire(repertoireProjet);
controleExistenceRepertoire(repertoireDonnees);
controleExistenceRepertoire(repertoireImages);

In [ ]:
def formatPct(pct, allvals):
    total = int(round(pct/100. * np.sum(allvals)))
    return "{:.2f}%\n({:d})".format(pct, total)   

In [ ]:
def affichageDistribution(colonne,couleur,ax, nom=''):
    graph = sns.distplot(colonne, color=couleur, ax=ax)
    graph.set(ylabel=None)
    moyenne, mediane = float(colonne.mean()), \
                   float(colonne.median())
    
    ax.axvline(moyenne, color='g', linestyle='-', label=f"{nom:12s} mean   = {moyenne:0.4f}", lw=2)
    ax.axvline(mediane, color='b', linestyle='--', label=f"{nom:12s} median = {mediane:0.4f}", lw=2)
    graph.legend(loc="upper right")

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Lecture des données</div></b>

<table>
    <tr> 
        <th  style="text-align:left">
            <table>
                <CAPTION style='padding:15px;color:#030aa7;font-size:150%;text-align: left;font-weight: bold;font-family: Georgia, serif'>Iris.csv</CAPTION>    
            <tr>                                                                                   
                <tr>                                                                                   
                     <th  style="text-align:left;background-color:#053061;color:white;">Colonne initiale </th>
                     <th  style="text-align:left;background-color:#053061;color:white;">Description</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Sex</th>
                    <th  style="text-align:left">M, F et I (nourrisson)</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Longueur(mm)</th>
                    <th  style="text-align:left">mesure de la coquille la plus longue</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Diametre(mm)</th>
                    <th  style="text-align:left">perpendiculaire à la longueur</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Hauteur(mm)</th>
                    <th  style="text-align:left">avec la chair dans la coquille</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Poids.Entier(grammes)</th>
                    <th  style="text-align:left">poids du ormeau entier</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Poids.Decortique(grammes)</th>
                    <th  style="text-align:left">poids de la chair</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Poids.Visceres(grammes)</th>
                    <th  style="text-align:left">poids des intestins (après saignée)</th>
                </tr>
                <tr>
                    <th  style="text-align:left">Poids.Coquille(grammes)</th>
                    <th  style="text-align:left">poids coquille après séchage</th>
                </tr>
                <tr>
                    <th  style="text-align:left;color:red;">Anneaux</th>
                    <th  style="text-align:left;color:red;">+1,5 donne l'âge en années</th>
                </tr>
            </table>
        </th>
        <th  style="text-align:left"><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/abalone01.jpg" width="512"></th>
    </tr>
</table>


In [ ]:
donnees = pd.read_csv("../donnees/abalone.csv").iloc[:,1:]
donnees.sample(5)

In [ ]:
donnees.columns

In [ ]:
qualitatives = ['Sex', 'Anneaux']
quantitatives = ['Longueur', 'Diametre', 'Hauteur', 'Poids.Entier', 'Poids.Decortique', 'Poids.Visceres', 'Poids.Coquille']

In [ ]:
donnees.Anneaux.sort_values().unique()

In [ ]:
donnees['Age'] = donnees.Anneaux + 1.5 
donnees['AgeQ'] = pd.qcut(donnees.Age, 6)

In [ ]:
donnees['AnneauxQ'] = pd.qcut(donnees.Anneaux, 4, labels=["petit", "moyen","grand","très grand"])

In [ ]:
donnees.describe()

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Statistiques descriptives et analyse de données</div></b>

In [ ]:
radius,size=0.8,0.3
fig,ax = plt.subplots(ncols=1,figsize=(16,16), subplot_kw=dict(aspect="equal"))

affichage = donnees.groupby('Sex').Anneaux.count().reset_index().rename(columns={'Anneaux':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage

wedges, texts, autotexts =  ax.pie(
         affichage['nombre'],
         autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
         labels=affichage.Sex.values,
         # shadow=True, 
         counterclock=False,
         startangle=90 ,
         colors = palette,
         # pctdistance=0.4, 
         labeldistance=1.1, 
         textprops=dict(color="#030aa7"),
         explode=[0.01 for _ in range(affichage.Sex.count())]
      );
plt.setp(autotexts, size=24, weight="bold",color="w")
plt.setp(texts, size=32, weight="bold");
ax.set_title("Sexe d’Ormeaux",fontdict=dict(color="#030aa7", size=56));

In [ ]:
radius,size=0.8,0.3
fig,ax = plt.subplots(ncols=1,figsize=(16,16), subplot_kw=dict(aspect="equal"))

affichage = donnees.groupby('AgeQ').Sex.count().reset_index().rename(columns={'Sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage

wedges, texts, autotexts =  ax.pie(
         affichage['nombre'],
         autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
         labels=affichage.AgeQ,
         # shadow=True, 
         counterclock=False,
         startangle=90 ,
         colors = palette,
         # pctdistance=0.4, 
         labeldistance=1.1, 
         textprops=dict(color="#030aa7"),
         explode=[0.01 for _ in range(affichage.AgeQ.count())]
      );
plt.setp(autotexts, size=24, weight="bold",color="w")
plt.setp(texts, size=32, weight="bold");
ax.set_title("Age d’Ormeaux",fontdict=dict(color="#030aa7", size=56));

In [ ]:
radius,size=0.8,0.3
fig,ax = plt.subplots(ncols=1,figsize=(16,16), subplot_kw=dict(aspect="equal"))

affichage = donnees.groupby('AnneauxQ').Sex.count().reset_index().rename(columns={'Sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage

wedges, texts, autotexts =  ax.pie(
         affichage['nombre'],
         autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
         labels=affichage.AnneauxQ,
         # shadow=True, 
         counterclock=False,
         startangle=90 ,
         colors = palette,
         # pctdistance=0.4, 
         labeldistance=1.1, 
         textprops=dict(color="#030aa7"),
         explode=[0.01 for _ in range(affichage.AnneauxQ.count())]
      );
plt.setp(autotexts, size=24, weight="bold",color="w")
plt.setp(texts, size=32, weight="bold");
ax.set_title("Anneaux d’Ormeaux",fontdict=dict(color="#030aa7", size=56));

In [ ]:
affichage = donnees.assign(AgeQ = donnees.AgeQ.astype(str)
                          ).groupby(['Sex', 'AgeQ']
                                   ).Anneaux.count().reset_index().rename(columns={'Anneaux':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage

fig = go.Figure(px.treemap(affichage, 
                           path=[px.Constant("Global"),'Sex', 'AgeQ'], 
                           values='nombre',
                           color='nombre', 
                           hover_data=['Sex', 'AgeQ'],
                  color_continuous_scale='RdBu',
                  color_continuous_midpoint=affichage['nombre'].mean(),
                 width=1152,
                 height=768
                ))
fig.show()

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Corrélation de Pearson</div></b>

<table>        
<tr>                                                                                   
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/rmse.png" ></th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/rse.png" ><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/correlation_pearson.png" ></th>
</tr> 
</table>
<img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/correlation_pearson_graphs.png" width="1024">

In [ ]:
plt.figure(figsize=(56,56))
sns.set(font_scale=3)
plt.title('Correlation Pearson des variables', y=1.05, size=64)
sns.heatmap(donnees[quantitatives+['Anneaux']].corr(),linewidths=0.3, fmt= '.2f', #vmax=1.0, 
            square=True, cmap='coolwarm', linecolor='white', annot=True)
# sauvegarderImage('Correlation Pearson des variables')   
sns.set(font_scale=2)

In [ ]:
donneesM = donnees.melt(id_vars=['Sex','Anneaux','AnneauxQ'], 
                        value_vars=['Longueur', 'Diametre', 'Hauteur', 'Poids.Entier',
                                    'Poids.Decortique', 'Poids.Visceres', 'Poids.Coquille'])

In [ ]:
donneesM.head()

In [ ]:
plt.figure(figsize=(32,24))
gbarplot = sns.barplot(x='variable',y='value',hue='Sex', data=donneesM,palette=palette,alpha=0.8,estimator='mean')

In [ ]:
plt.figure(figsize=(32,24))
gbarplot = sns.barplot(x='variable',y='value',hue='AnneauxQ', data=donneesM,palette=palette,alpha=0.8,estimator='mean')

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Les distributions des variables quantitatives</div></b>

In [ ]:
for colonne in quantitatives:
    plt.figure(figsize=(32,32))
    plt.title(colonne)
    sns.distplot(donnees[colonne][donnees.Sex == 'M'],color=palette[0], label='M', hist_kws=dict(alpha=0.4),bins=30)
    sns.distplot(donnees[colonne][donnees.Sex == 'F'],color=palette[1], label='F', hist_kws=dict(alpha=0.4),bins=30)
    sns.distplot(donnees[colonne][donnees.Sex == 'I'],color=palette[2], label='I', hist_kws=dict(alpha=0.4),bins=30)
    plt.legend();    

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.distplot(donnees[colonne][donnees.Sex == 'M'],color=palette[0], label='M', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.Sex == 'F'],color=palette[1], label='F', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.Sex == 'I'],color=palette[2], label='I', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])  
    ax[i].legend() 

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'petit'],color=palette[0], label='petit', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'moyen'],color=palette[1], label='moyen', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'grand'],color=palette[2], label='grand', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])    
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'très grand'],color=palette[3], label='très grand', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])    
    ax[i].legend()

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.boxplot(x=colonne,data=donnees,hue='Sex', palette=palette, ax=ax[i]);

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.boxplot(x=colonne,data=donnees,hue='AnneauxQ', palette=palette, ax=ax[i]);

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Pair Plot</div></b>

In [ ]:
affichage = donnees[quantitatives].copy()
affichage['Sex'] = donnees['Sex']
sns.set(font_scale=5)
graph = sns.pairplot(
             affichage,
             hue='Sex', 
             size=12, 
             aspect=1, 
             palette=palette,
             plot_kws={"s": 1200,"alpha":0.6}, 
             markers=["o", "s", "^"],   
             corner=True, 
             diag_kind="kde")
graph.map_upper(sns.kdeplot, levels=24, color=".2");
graph._legend.remove()
graph.add_legend(fontsize='xx-large', title_fontsize='xx-large');
sns.set(font_scale=3)
# sauvegarderImage('SantéFœtale-pairplot')

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Centrage et réduction des données</div></b>
<table>
<tr>
<th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/moyenne.png"></th>
<th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/ecart_type.png"></th>
<th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/centrage_reduction.png"></th>
</tr>
</table>

In [ ]:
from sklearn.preprocessing import StandardScaler
modelStd = StandardScaler()
# modelStd.fit(donnees[quantitatives])
# donnees[quantitatives] = modelStd.transform(donnees[quantitatives])
donnees[quantitatives] = modelStd.fit_transform(donnees[quantitatives])

In [ ]:
donneesM = donnees.melt(id_vars=['Sex','Anneaux','AnneauxQ'], value_vars=['Longueur', 'Diametre', 'Hauteur', 'Poids.Entier',
                                    'Poids.Decortique', 'Poids.Visceres', 'Poids.Coquille'])

In [ ]:
donneesM.head()

In [ ]:
plt.figure(figsize=(32,24))
gbarplot = sns.barplot(x='variable',y='value',hue='Sex', data=donneesM,palette=palette,alpha=0.8,estimator='mean')

In [ ]:
plt.figure(figsize=(32,24))
gbarplot = sns.barplot(x='variable',y='value',hue='AnneauxQ', data=donneesM,palette=palette,alpha=0.8,estimator='mean')

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.distplot(donnees[colonne][donnees.Sex == 'M'],color=palette[0], label='M', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.Sex == 'F'],color=palette[1], label='F', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.Sex == 'I'],color=palette[2], label='I', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])  
    ax[i].legend() 

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'petit'],color=palette[0], label='petit', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'moyen'],color=palette[1], label='moyen', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'grand'],color=palette[2], label='grand', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])    
    sns.distplot(donnees[colonne][donnees.AnneauxQ == 'très grand'],color=palette[3], label='très grand', hist_kws=dict(alpha=0.4),bins=30, ax=ax[i])    
    ax[i].legend()

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.boxplot(y=colonne,data=donnees,hue='Sex', palette=palette, ax=ax[i]);

In [ ]:
fig,ax = plt.subplots(1,7,figsize=(84,12));

for i,colonne in enumerate(quantitatives):
    sns.boxplot(y=colonne,data=donnees,hue='AnneauxQ', palette=palette, ax=ax[i]);